In [14]:
filename = 'wdsweb_summ2.txt'

# column format per https://www.astro.gsu.edu/wds/Webtextfiles/wdsweb_format.txt
columns = [
    (1,10,'s','id'),             #           2000 Coordinates
    (11,17,'s','discoverer'),    #           Discoverer & Number
    (18,22,'s','components'),    #           Components
    (24,27,'i','first_date'),    #           Date (first)
    (29,32,'i','last_date'),     #           Date (last)
    (34,37,'i','num_obs'),       #           Number of Observations (up to 9999)
    (39,41,'i','first_pa'),      #           Position Angle (first - XXX)
    (43,45,'i','last_pa'),       #           Position Angle (last  - XXX)
    (47,51,'f','first_sep'),     #         Separation (first)
    (53,57,'f','last_sep'),      #         Separation (last)
    (59,63,'f','pri_mag'),       #         Magnitude of First Component
    (65,69,'f','sec_mag'),       #         Magnitude of Second Component
    (71,79,'s','spectral'),      #         Spectral Type (Primary/Secondary)
    (81,84,'i','pri_pm_ra'),     #         Primary Proper Motion (RA)
    (85,88,'i','pri_pm_dec'),    #         Primary Proper Motion (Dec)
    (90,93,'i','sec_pm_ra'),     #         Secondary Proper Motion (RA)
    (94,97,'i','sec_pm_dec'),    #         Secondary Proper Motion (Dec)
    (99,106,'s','dm'),           #         Durchmusterung Number
    (108,111,'s','notes'),       #         Notes
    (113,130,'s','j2000'),       #        2000 arcsecond coordinates
]
header_lines = 4

# parse the file and return an array of rows
def parse(filename):
   with open(filename, "r") as file:
      line_num = 0
      rows = []
      #read the file line by line
      for line in file:
         line_num += 1
         #skip the header
         if line_num <= header_lines: 
            continue

         row = parse_line(line_num, line)
         rows.append(row)
      
      return rows   

# parse a line and return a dictionary of fields
def parse_line(line_num, line):
   row = {}
   for column in columns:
      (start,end,format,key) = column
      if end<=len(line):
         data_str = line[start-1:end]
         data = parse_field(line_num, data_str, format)
         row[key]=data

   return row

# parse the string field data with the given format
def parse_field(line_num, data_str, format):
   data_str = data_str.rstrip()
   # empty strings stay empty strings
   match(format):
      case 's':
         return data_str
      # conversion errors produce NaN
      case 'i':
         try:
            return int(data_str)
         except Exception as e:
            return float('nan')
      case 'f':
         try:
            return float(data_str)
         except Exception as e:
            return float('nan')

      # unknown formats are empty strings
      case _:
         return ""
         

data = parse(filename)


In [15]:
from astropy.table import Table

print(data[0])
# load data into an astropy table
table = Table(rows=data)

# save table in HDF5 format
filename = "wdsweb.hdf5"
table.write(filename,serialize_meta=True,overwrite=True)


{'id': '00000+7530', 'discoverer': 'A  1248', 'components': '', 'first_date': 1904, 'last_date': 1982, 'num_obs': 5, 'first_pa': 246, 'last_pa': 235, 'first_sep': 0.8, 'last_sep': 0.6, 'pri_mag': 10.27, 'sec_mag': 11.5, 'spectral': 'A7IV', 'pri_pm_ra': 34, 'pri_pm_dec': 5, 'sec_pm_ra': nan, 'sec_pm_dec': nan, 'dm': '+74 1056', 'notes': '', 'j2000': '000006.64+752859.8'}


/home/adamarthurryan/micromamba/envs/jupyter/lib/python3.13/site-packages/astropy/io/misc/hdf5.py:282: UserWarning: table path was not set via the path= argument; using default path __astropy_table__
  warnings.warn(
